# Sesión 2 — Demo: RAG sobre Reviews de Electrónica

Notebook de demostración para la **Sesión 2** del curso de IA Generativa (ISDI MDA).

**Contenido:**
1. Configuración del modelo (Gemini o Ollama — una sola celda)
2. Carga de datos: 300 reviews de Amazon Electronics
3. Construcción del índice vectorial en ChromaDB
4. Baseline: preguntar al LLM sin contexto
5. RAG: preguntar con contexto recuperado
6. Comparación lado a lado

---

> **Backend `gemini`**: ejecutar en Google Colab con una API key en Secrets.  
> **Backend `ollama`**: ejecutar en local con Ollama corriendo (`ollama serve`).  
> El resto del notebook no cambia.

In [1]:
%pip install -q \
    langchain \
    langchain-google-genai \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    "datasets<3.0" \
    pandas


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Configuración del modelo

Cambiad `BACKEND` para alternar entre Gemini y Ollama. **Solo esta celda diferencia los dos entornos.**

In [2]:
# ── Configuración ────────────────────────────────────────────────────────────
BACKEND = "gemini"          # "gemini" | "ollama"
OLLAMA_MODEL = "qwen3.5:2b"
OLLAMA_EMBEDDING_MODEL = "nomic-embed-text-v2-moe"
# ─────────────────────────────────────────────────────────────────────────────

if BACKEND == "gemini":
    GOOGLE_API_KEY = "your-api-key-here"

    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY
    )
    print("✓ Backend: Gemini 2.5 Flash")

elif BACKEND == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL)
    
    print(f"✓ Backend: Ollama — {OLLAMA_MODEL}")

else:
    raise ValueError(f"Backend desconocido: {BACKEND}. Usa 'gemini' o 'ollama'.")

from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)
print(f"✓ Backend: Ollama — {OLLAMA_EMBEDDING_MODEL}")

✓ Backend: Gemini 2.5 Flash
✓ Backend: Ollama — nomic-embed-text-v2-moe


## 2. Carga de datos

Usamos las primeras 300 reviews del dataset de Amazon Electronics.

In [3]:
from datasets import load_dataset
import pandas as pd

print("Cargando dataset...")
ds = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Electronics",
    split="full",
    streaming=True,
    trust_remote_code=True,
)
df = pd.DataFrame(ds.take(300))[["title", "text", "rating", "parent_asin"]].dropna(subset=["text"])
df["rating"] = df["rating"].astype(int)

print(f"✓ {len(df)} reviews cargadas")
print(f"Distribución de ratings:\n{df['rating'].value_counts().sort_index()}")
print(f"\nEjemplo:")
print(df.iloc[0][["rating", "title", "text"]])

Cargando dataset...
✓ 300 reviews cargadas
Distribución de ratings:
rating
1     24
2      8
3     24
4     48
5    196
Name: count, dtype: int64

Ejemplo:
rating                                                    3
title                     Smells like gasoline! Going back!
text      First & most offensive: they reek of gasoline ...
Name: 0, dtype: object


## 3. Construcción del índice vectorial

Cada review se convierte en un `Document` de LangChain y se indexa en ChromaDB.  
La metadata (`rating`, `parent_asin`) permite filtrar después.

In [4]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

# Convertir filas del dataframe a Documents
docs = [
    Document(
        page_content=f"{row['title']}\n\n{row['text']}",
        metadata={
            "rating": row["rating"],
            "parent_asin": row["parent_asin"]
        }
    )
    for _, row in df.iterrows()
]

print(f"Indexando {len(docs)} documentos en ChromaDB...")
print("(Puede tardar 1-2 minutos mientras se generan los embeddings)")

vectorstore = Chroma.from_documents(docs, embeddings)

print(f"✓ Índice creado con {vectorstore._collection.count()} vectores")

Indexando 300 documentos en ChromaDB...
(Puede tardar 1-2 minutos mientras se generan los embeddings)
✓ Índice creado con 300 vectores


## 4. Baseline: preguntar al LLM sin contexto

Preguntamos directamente al modelo, sin ninguna información del catálogo.  
El modelo responde solo con lo que aprendió durante el entrenamiento.

In [5]:
PREGUNTA = "¿Qué problemas comunes tienen los auriculares inalámbricos según los clientes?"

respuesta_sin_rag = llm.invoke(PREGUNTA)

print("=== SIN RAG ===")
print(respuesta_sin_rag.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}

## 5. RAG: preguntar con contexto recuperado

La cadena RAG recupera las reviews más relevantes y las inyecta en el prompt antes de generar la respuesta.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", """Eres un asistente experto en análisis de productos de electrónica.
Responde usando SOLO la información de las siguientes reviews de clientes reales.
Si la información no aparece en las reviews, dilo explícitamente.
Cita ejemplos concretos de las reviews cuando sea relevante.

Reviews recuperadas:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('rating', '?')}★] {d.page_content[:400]}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

print("✓ Cadena RAG construida")

In [ ]:
respuesta_con_rag = rag_chain.invoke(PREGUNTA)

print("=== CON RAG ===")
print(respuesta_con_rag)

In [ ]:
rag_chain.invoke("Me das una receta para hacer torrijas?")

## 6. Comparación y exploración

Veamos qué documentos recuperó el retriever, y probemos otras preguntas.

In [ ]:
# Inspeccionar los documentos recuperados
docs_recuperados = retriever.invoke(PREGUNTA)

print(f"Documentos recuperados para: '{PREGUNTA}'\n")
for i, doc in enumerate(docs_recuperados, 1):
    print(f"--- Documento {i} [{doc.metadata.get('rating', '?')}★] ---")
    print(doc.page_content[:250])
    print()

In [ ]:
# Otras preguntas para explorar
otras_preguntas = [
    "¿Qué productos tienen mejor valoración en calidad de sonido?",
    "¿Los clientes mencionan problemas con la duración de la batería?",
    "¿Qué dicen los clientes sobre el servicio de entrega?",
]

for pregunta in otras_preguntas:
    print(f"\nP: {pregunta}")
    print(f"R: {rag_chain.invoke(pregunta)[:300]}...")
    print("-" * 60)

In [ ]:
# Comparación final: misma pregunta, dos enfoques
pregunta_final = "¿Vale la pena comprar un cable USB-C de esta marca?"

print("PREGUNTA:", pregunta_final)
print()
print("SIN RAG:")
print(llm.invoke(pregunta_final).content[:400])
print()
print("CON RAG:")
print(rag_chain.invoke(pregunta_final)[:400])

---

## Cambio de backend: Ollama

Para demostrar la independencia del pipeline respecto al proveedor:

1. Volved a la celda de configuración (celda 2)
2. Cambiad `BACKEND = "ollama"`
3. Ejecutad desde ahí hasta el final

El resto del código no cambia. El pipeline RAG es idéntico; solo cambia el objeto `llm`.

> **Requisito**: Ollama corriendo localmente con `ollama serve` y los modelos descargados:
> ```bash
> ollama pull qwen3:4b
> ollama pull nomic-embed-text-v2-moe
> ```